In [15]:
import pandas as pd

in_path = "/Users/aishwarya/Desktop/ambiguous.csv"   # exported from Numbers
out_path = "/Users/aishwarya/Desktop/amb.txt"

df = pd.read_csv(in_path, sep=";", header=None)

# keep only the first two numeric columns
ra = df.iloc[:, 1]
dec = df.iloc[:, 2]

clean = pd.DataFrame({"RA": ra, "DEC": dec}).dropna()

clean.to_csv(out_path, sep=" ", header=False, index=False)
print(f"Saved to {out_path}")


Saved to /Users/aishwarya/Desktop/amb.txt


In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

from astropy.io import fits
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
from astropy.nddata import Cutout2D
from astropy.coordinates import SkyCoord
import astropy.units as u

from astropy.convolution import convolve, Gaussian2DKernel
from astropy.visualization import ZScaleInterval


# -------------------------------------------------
# USER INPUT
# -------------------------------------------------
txt_file = "/Users/aishwarya/Desktop/CSV/Final_sources_ra_dec.txt"

fits_files = {
    "Y": "/Users/aishwarya/Desktop/Lyman_alpha_2/Background_check/BG_Y_Y.fits",
    "Z": "/Users/aishwarya/Desktop/Lyman_alpha_2/Background_check/BG_Y_Z.fits",
    "I": "/Users/aishwarya/Desktop/Lyman_alpha_2/Background_check/BG_Y_I.fits"
}

output_pdf = "/Users/aishwarya/Desktop/source_cutouts.pdf"

cutout_size_arcsec = 10        # cutout size (arcsec)
smooth_fwhm_arcsec = 0.6      # DS9-like smoothing (FWHM in arcsec)


# -------------------------------------------------
# LOAD TXT FILE (NO HEADER)
# -------------------------------------------------
df = pd.read_csv(
    txt_file,
    delim_whitespace=True,
    header=None
)

df.columns = ["RA", "DEC"]

print("\nLoaded sources:")
print(df.head())


# -------------------------------------------------
# LOAD FITS + WCS
# -------------------------------------------------
images = {}
wcs_dict = {}

for band, path in fits_files.items():
    with fits.open(path) as hdul:
        images[band] = hdul[0].data
        wcs_dict[band] = WCS(hdul[0].header)


# -------------------------------------------------
# CREATE PDF
# -------------------------------------------------
with PdfPages(output_pdf) as pdf:

    for idx, row in df.iterrows():

        ra = float(row["RA"])
        dec = float(row["DEC"])

        coord = SkyCoord(ra*u.deg, dec*u.deg)

        fig, axes = plt.subplots(1, 3, figsize=(10, 3))

        for ax, (band, data) in zip(axes, images.items()):

            wcs = wcs_dict[band]

            # Pixel scale (arcsec per pixel)
            pix_scales = proj_plane_pixel_scales(wcs) * 3600
            pixel_scale = np.mean(pix_scales)

            size_pix = int(cutout_size_arcsec / pixel_scale)

            try:
                cutout = Cutout2D(
                    data,
                    position=coord,
                    size=(size_pix, size_pix),
                    wcs=wcs,
                    mode="partial",
                    fill_value=np.nan
                )
            except Exception:
                print(f"Skipping source {idx} (outside image)")
                continue

            # -------------------------------------------------
            # DS9-LIKE SMOOTHING (Gaussian FWHM in arcsec)
            # -------------------------------------------------
            sigma_pix = (smooth_fwhm_arcsec / 2.355) / pixel_scale
            kernel = Gaussian2DKernel(sigma_pix)

            smooth_data = convolve(
                cutout.data,
                kernel,
                normalize_kernel=True,
                nan_treatment='interpolate'
            )

            # -------------------------------------------------
            # DS9-LIKE ZSCALE CONTRAST
            # -------------------------------------------------
            interval = ZScaleInterval()
            vmin, vmax = interval.get_limits(smooth_data)

            ax.imshow(
                smooth_data,
                origin="lower",
                cmap="gray",
                vmin=vmin,
                vmax=vmax
            )

            ax.set_title(f"{band}-band", fontsize=11)
            ax.axis("off")

            # Red crosshair
            center = smooth_data.shape[0] // 2
            ax.plot([center-6, center+6], [center, center], color='red', lw=1)
            ax.plot([center, center], [center-6, center+6], color='red', lw=1)

        fig.suptitle(
            f"RA = {ra:.6f}°    DEC = {dec:.6f}°",
            fontsize=12
        )

        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

print("\nPDF saved at:")
print(output_pdf)


/var/folders/9v/db3j63px68q3f33kdb7_7wbc0000gn/T/ipykernel_11542/575591117.py:37: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(



Loaded sources:
           RA        DEC
0  358.193029 -30.942987
1  357.459842 -31.629391
2  357.278628 -31.660727
3  356.413992 -31.564850
4  356.170926 -31.317514

PDF saved at:
/Users/aishwarya/Desktop/source_cutouts.pdf


In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

from astropy.io import fits
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
from astropy.nddata import Cutout2D
from astropy.coordinates import SkyCoord
import astropy.units as u

from astropy.convolution import convolve, Gaussian2DKernel
from astropy.visualization import ZScaleInterval


# -------------------------------------------------
# USER INPUT
# -------------------------------------------------
txt_file = "/Users/aishwarya/Desktop/amb.txt"

fits_files = {
    "Y": "/Users/aishwarya/Desktop/Lyman_alpha_2/Background_check/BG_Y_Y.fits",
    "Z": "/Users/aishwarya/Desktop/Lyman_alpha_2/Background_check/BG_Y_Z.fits",
    "I": "/Users/aishwarya/Desktop/Lyman_alpha_2/Background_check/BG_Y_I.fits"
}

output_pdf = "/Users/aishwarya/Desktop/amb_cutouts.pdf"

cutout_size_arcsec = 10        # cutout size (arcsec)
smooth_fwhm_arcsec = 0.6      # DS9-like smoothing (FWHM in arcsec)


# -------------------------------------------------
# LOAD TXT FILE (NO HEADER)
# -------------------------------------------------
df = pd.read_csv(
    txt_file,
    delim_whitespace=True,
    header=None
)

df.columns = ["RA", "DEC"]

print("\nLoaded sources:")
print(df.head())


# -------------------------------------------------
# LOAD FITS + WCS
# -------------------------------------------------
images = {}
wcs_dict = {}

for band, path in fits_files.items():
    with fits.open(path) as hdul:
        images[band] = hdul[0].data
        wcs_dict[band] = WCS(hdul[0].header)


# -------------------------------------------------
# CREATE PDF
# -------------------------------------------------
with PdfPages(output_pdf) as pdf:

    for idx, row in df.iterrows():

        ra = float(row["RA"])
        dec = float(row["DEC"])

        coord = SkyCoord(ra*u.deg, dec*u.deg)

        fig, axes = plt.subplots(1, 3, figsize=(10, 3))

        for ax, (band, data) in zip(axes, images.items()):

            wcs = wcs_dict[band]

            # Pixel scale (arcsec per pixel)
            pix_scales = proj_plane_pixel_scales(wcs) * 3600
            pixel_scale = np.mean(pix_scales)

            size_pix = int(cutout_size_arcsec / pixel_scale)

            try:
                cutout = Cutout2D(
                    data,
                    position=coord,
                    size=(size_pix, size_pix),
                    wcs=wcs,
                    mode="partial",
                    fill_value=np.nan
                )
            except Exception:
                print(f"Skipping source {idx} (outside image)")
                continue

            # -------------------------------------------------
            # DS9-LIKE SMOOTHING (Gaussian FWHM in arcsec)
            # -------------------------------------------------
            sigma_pix = (smooth_fwhm_arcsec / 2.355) / pixel_scale
            kernel = Gaussian2DKernel(sigma_pix)

            smooth_data = convolve(
                cutout.data,
                kernel,
                normalize_kernel=True,
                nan_treatment='interpolate'
            )

            # -------------------------------------------------
            # DS9-LIKE ZSCALE CONTRAST
            # -------------------------------------------------
            interval = ZScaleInterval()
            vmin, vmax = interval.get_limits(smooth_data)

            ax.imshow(
                smooth_data,
                origin="lower",
                cmap="gray",
                vmin=vmin,
                vmax=vmax
            )

            ax.set_title(f"{band}-band", fontsize=11)
            ax.axis("off")

            # Red crosshair
            center = smooth_data.shape[0] // 2
            ax.plot([center-6, center+6], [center, center], color='red', lw=1)
            ax.plot([center, center], [center-6, center+6], color='red', lw=1)

        fig.suptitle(
            f"RA = {ra:.6f}°    DEC = {dec:.6f}°",
            fontsize=12
        )

        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

print("\nPDF saved at:")
print(output_pdf)


/var/folders/9v/db3j63px68q3f33kdb7_7wbc0000gn/T/ipykernel_37037/1930898839.py:37: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(



Loaded sources:
           RA        DEC
0  358.193029 -30.942987
1  358.041260 -31.299306
2  357.459842 -31.629391
3  357.278628 -31.660727
4  356.946579 -31.611600

PDF saved at:
/Users/aishwarya/Desktop/amb_cutouts.pdf


In [4]:
import pandas as pd

df = pd.read_csv("/Users/aishwarya/Desktop/sources.csv")

print(df.columns.tolist())


[';;;;;;']


# cdfs

# remove amb sources

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

from astropy.io import fits
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
from astropy.nddata import Cutout2D
from astropy.coordinates import SkyCoord
import astropy.units as u

from astropy.convolution import convolve, Gaussian2DKernel
from astropy.visualization import ZScaleInterval


# -------------------------------------------------
# USER INPUT
# -------------------------------------------------
txt_file = "/Users/aishwarya/Desktop/source_cdfs_true.txt"

fits_files = {
    "Y": "/Users/aishwarya/Desktop/new_cdfs/trimmed_2deg/trim2deg_y_sci.fits",
    "Z": "/Users/aishwarya/Desktop/new_cdfs/trimmed_2deg/trim2deg_z_sci.fits",
    "I": "/Users/aishwarya/Desktop/new_cdfs/trimmed_2deg/trim2deg_i_sci.fits"
}

output_pdf = "/Users/aishwarya/Desktop/source_cutouts_cdfs.pdf"

cutout_size_arcsec = 10        # cutout size (arcsec)
smooth_fwhm_arcsec = 0.6      # DS9-like smoothing (FWHM in arcsec)


# -------------------------------------------------
# LOAD TXT FILE (NO HEADER)
# -------------------------------------------------
df = pd.read_csv(
    txt_file,
    sep="\t",          # tab separated
    header=None,       # no column names in file
    comment="#"
)

# assign column names
df.columns = ["RA", "DEC"]



print("\nLoaded sources:")
print(df.head())


# -------------------------------------------------
# LOAD FITS + WCS
# -------------------------------------------------
images = {}
wcs_dict = {}

for band, path in fits_files.items():
    with fits.open(path) as hdul:
        images[band] = hdul[0].data
        wcs_dict[band] = WCS(hdul[0].header)


# -------------------------------------------------
# CREATE PDF
# -------------------------------------------------
with PdfPages(output_pdf) as pdf:

    for idx, row in df.iterrows():

        ra = float(row["RA"])
        dec = float(row["DEC"])

        coord = SkyCoord(ra*u.deg, dec*u.deg)

        fig, axes = plt.subplots(1, 3, figsize=(10, 3))

        for ax, (band, data) in zip(axes, images.items()):

            wcs = wcs_dict[band]

            # Pixel scale (arcsec per pixel)
            pix_scales = proj_plane_pixel_scales(wcs) * 3600
            pixel_scale = np.mean(pix_scales)

            size_pix = int(cutout_size_arcsec / pixel_scale)

            try:
                cutout = Cutout2D(
                    data,
                    position=coord,
                    size=(size_pix, size_pix),
                    wcs=wcs,
                    mode="partial",
                    fill_value=np.nan
                )
            except Exception:
                print(f"Skipping source {idx} (outside image)")
                continue

            # -------------------------------------------------
            # DS9-LIKE SMOOTHING (Gaussian FWHM in arcsec)
            # -------------------------------------------------
            sigma_pix = (smooth_fwhm_arcsec / 2.355) / pixel_scale
            kernel = Gaussian2DKernel(sigma_pix)

            smooth_data = convolve(
                cutout.data,
                kernel,
                normalize_kernel=True,
                nan_treatment='interpolate'
            )

            # -------------------------------------------------
            # DS9-LIKE ZSCALE CONTRAST
            # -------------------------------------------------
            interval = ZScaleInterval()
            vmin, vmax = interval.get_limits(smooth_data)

            ax.imshow(
                smooth_data,
                origin="lower",
                cmap="gray",
                vmin=vmin,
                vmax=vmax
            )

            ax.set_title(f"{band}-band", fontsize=11)
            ax.axis("off")

            # Red crosshair
            center = smooth_data.shape[0] // 2
            ax.plot([center-6, center+6], [center, center], color='red', lw=1)
            ax.plot([center, center], [center-6, center+6], color='red', lw=1)

        fig.suptitle(
            f"RA = {ra:.6f}°    DEC = {dec:.6f}°",
            fontsize=12
        )

        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

print("\nPDF saved at:")
print(output_pdf)



Loaded sources:
          RA        DEC
0  52.309912 -28.542686
1  53.242619 -27.695620
2  52.826893 -27.430345
3  52.861144 -27.211088
4  51.985858 -27.852264

PDF saved at:
/Users/aishwarya/Desktop/source_cutouts_cdfs.pdf


In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

from astropy.io import fits
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
from astropy.nddata import Cutout2D
from astropy.coordinates import SkyCoord
import astropy.units as u
from matplotlib.patches import Circle

from astropy.convolution import convolve, Gaussian2DKernel
from astropy.visualization import ZScaleInterval


# -------------------------------------------------
# USER INPUT
# -------------------------------------------------
txt_file = "/Users/aishwarya/Desktop/amb_cdfs_true.txt"

fits_files = {
    "Y": "/Users/aishwarya/Desktop/new_cdfs/trimmed_2deg/trim2deg_y_sci.fits",
    "Z": "/Users/aishwarya/Desktop/new_cdfs/trimmed_2deg/trim2deg_z_sci.fits",
    "I": "/Users/aishwarya/Desktop/new_cdfs/trimmed_2deg/trim2deg_i_sci.fits"
}

output_pdf = "/Users/aishwarya/Desktop/source_cutouts_cdfs_AMB.pdf"

cutout_size_arcsec = 10        # cutout size (arcsec)
smooth_fwhm_arcsec = 0.6      # DS9-like smoothing (FWHM in arcsec)


# -------------------------------------------------
# LOAD TXT FILE (NO HEADER)
# -------------------------------------------------
df = pd.read_csv(
    txt_file,
    sep="\t",          # tab separated
    header=None,       # no column names in file
    comment="#"
)

# assign column names
df.columns = ["RA", "DEC"]



print("\nLoaded sources:")
print(df.head())


# -------------------------------------------------
# LOAD FITS + WCS
# -------------------------------------------------
images = {}
wcs_dict = {}

for band, path in fits_files.items():
    with fits.open(path) as hdul:
        images[band] = hdul[0].data
        wcs_dict[band] = WCS(hdul[0].header)


# -------------------------------------------------
# CREATE PDF
# -------------------------------------------------
with PdfPages(output_pdf) as pdf:

    for idx, row in df.iterrows():

        ra = float(row["RA"])
        dec = float(row["DEC"])

        coord = SkyCoord(ra*u.deg, dec*u.deg)

        fig, axes = plt.subplots(1, 3, figsize=(10, 3))

        for ax, (band, data) in zip(axes, images.items()):

            wcs = wcs_dict[band]

            # Pixel scale (arcsec per pixel)
            pix_scales = proj_plane_pixel_scales(wcs) * 3600
            pixel_scale = np.mean(pix_scales)

            size_pix = int(cutout_size_arcsec / pixel_scale)

            try:
                cutout = Cutout2D(
                    data,
                    position=coord,
                    size=(size_pix, size_pix),
                    wcs=wcs,
                    mode="partial",
                    fill_value=np.nan
                )
            except Exception:
                print(f"Skipping source {idx} (outside image)")
                continue

            # -------------------------------------------------
            # DS9-LIKE SMOOTHING (Gaussian FWHM in arcsec)
            # -------------------------------------------------
            sigma_pix = (smooth_fwhm_arcsec / 2.355) / pixel_scale
            kernel = Gaussian2DKernel(sigma_pix)

            smooth_data = convolve(
                cutout.data,
                kernel,
                normalize_kernel=True,
                nan_treatment='interpolate'
            )

            # -------------------------------------------------
            # DS9-LIKE ZSCALE CONTRAST
            # -------------------------------------------------
            interval = ZScaleInterval()
            vmin, vmax = interval.get_limits(smooth_data)

            ax.imshow(
                smooth_data,
                origin="lower",
                cmap="gray",
                vmin=vmin,
                vmax=vmax
            )

            ax.set_title(f"{band}-band", fontsize=11)
            ax.axis("off")

            # Red crosshair
            center = smooth_data.shape[0] // 2
            ax.plot([center-6, center+6], [center, center], color='red', lw=1)
            ax.plot([center, center], [center-6, center+6], color='red', lw=1)

# 2 arcsec circle overlay
            radius_pix = 2 / pixel_scale  # 2 arcsec radius → pixels
            circle = Circle(
                (center, center),
                radius=radius_pix,
                edgecolor='red',
                facecolor='none',
                lw=1.2
            )
            ax.add_patch(circle)

        fig.suptitle(
            f"RA = {ra:.6f}°    DEC = {dec:.6f}°",
            fontsize=12
        )

        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

print("\nPDF saved at:")
print(output_pdf)



Loaded sources:
          RA        DEC
0  52.041550 -28.464706
1  51.883174 -28.540355
2  52.768174 -28.416660
3  52.837793 -28.369643
4  53.141241 -28.401831



PDF saved at:
/Users/aishwarya/Desktop/source_cutouts_cdfs_AMB.pdf


# make region files 

In [10]:

import numpy as np
import os

# ---------------------------------------
# Files to convert
# ---------------------------------------
files = [
    "/Users/aishwarya/Desktop/comparision/amb.txt",
    "/Users/aishwarya/Desktop/comparision/source.txt"
]

# ---------------------------------------
# Convert catalog to DS9 region
# ---------------------------------------
def txt_to_reg(file):

    # Read RA DEC (no header)
    data = np.loadtxt(file)

    ra = data[:, 0]
    dec = data[:, 1]

    # Output region file
    reg_file = file.replace(".txt", "_points.reg")

    with open(reg_file, "w") as f:

        f.write("# Region file format: DS9 version 4.1\n")
        f.write("global color=cyan width=2\n")
        f.write("fk5\n")

        for r, d in zip(ra, dec):
            f.write(f"point({r},{d}) # point=circle 8\n")

    print(f"Created: {reg_file}")


# ---------------------------------------
# Run conversion
# ---------------------------------------
for file in files:

    if not os.path.exists(file):
        print(f"File not found: {file}")
        continue

    txt_to_reg(file)

print("All files converted.")



Created: /Users/aishwarya/Desktop/comparision/amb_points.reg
Created: /Users/aishwarya/Desktop/comparision/source_points.reg
All files converted.


In [7]:
import pandas as pd
import numpy as np

# ============================================================
# FILES
# ============================================================
main_file = "/Users/aishwarya/Desktop/source_cdfs_RADEC.cat"
amb_file = "/Users/aishwarya/Desktop/amb_cdfs.txt"
output_file = "/Users/aishwarya/Desktop/source_cdfs_clean.cat"

# Matching tolerance (in degrees)
# 1 arcsec = 1/3600 deg
tolerance = 1.0 / 3600   # 1 arcsec

# ============================================================
# LOAD FILES
# ============================================================
main_df = pd.read_csv(main_file, sep=r"\s+", comment="#")
amb_df = pd.read_csv(amb_file, sep=r"\s+", header=None)

amb_df.columns = ['ALPHA_J2000', 'DELTA_J2000']

# Keep only RA/Dec from main
main_df = main_df[['ALPHA_J2000', 'DELTA_J2000']]

print(f"Main sources: {len(main_df)}")
print(f"Ambiguous sources: {len(amb_df)}")

# ============================================================
# REMOVE MATCHES
# ============================================================
def is_match(ra, dec, amb_df, tol):
    return np.any(
        (np.abs(amb_df['ALPHA_J2000'] - ra) < tol) &
        (np.abs(amb_df['DELTA_J2000'] - dec) < tol)
    )

mask = []

for _, row in main_df.iterrows():
    ra = row['ALPHA_J2000']
    dec = row['DELTA_J2000']

    match = is_match(ra, dec, amb_df, tolerance)
    mask.append(not match)   # keep only NON-matching

clean_df = main_df[mask]

print(f"Remaining sources after removal: {len(clean_df)}")

# ============================================================
# SAVE OUTPUT
# ============================================================
clean_df.to_csv(output_file, sep=' ', index=False)

print(f"Saved cleaned catalog: {output_file}")

Main sources: 38
Ambiguous sources: 19
Remaining sources after removal: 19
Saved cleaned catalog: /Users/aishwarya/Desktop/source_cdfs_clean.cat
